# Bead Mosaic — Phase 2
Zero-config demo: run all cells top-to-bottom.
To use your own photos: set `CONFIG['BASE_DIR']` + `CONFIG['TARGET_PATH']` in Cell 2 and flip `DEMO_MODE` to `False`.

**Phase 2 adds three knobs:**
- `LAMBDA` — repetition penalty (0 = no penalty, large = force tile diversity)
- `MU` — neighbor penalty (0 = ignore, large = no adjacent duplicates)
- `TAU` — tone transfer strength [0, 1] (0 = original tile colors, 1 = tiles adopt target patch's LAB mean)


In [ ]:
%pip install -q -r requirements.txt
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import ipywidgets as widgets
from PIL import Image, ImageOps
from skimage import data
from skimage.color import rgb2lab
from IPython.display import display

from src import match, render, scan

In [ ]:
# --- edit these for real usage ---
CONFIG = {
    'BASE_DIR': Path.cwd() / 'my_photos',
    'TARGET_PATH': None,  # None -> use skimage astronaut
    'GRID_W': 120,
    'GRID_H': 68,
    'TILE_PX': 24,
    'CACHE_DIR': Path.cwd() / '.cache',
    'OUTPUT_PATH': Path.cwd() / 'output.png',
    'DEMO_MODE': True,
    'LAMBDA': 0.0,  # repetition penalty
    'MU': 0.0,      # neighbor penalty
    'TAU': 0.0,     # tone transfer strength [0, 1]
}
CONFIG

In [ ]:
pool = scan.build_pool(
    base_dir=CONFIG['BASE_DIR'],
    cache_dir=CONFIG['CACHE_DIR'],
    tile_px=CONFIG['TILE_PX'],
    demo_mode=CONFIG['DEMO_MODE'],
)
print(f"pool size: {pool.lab.shape[0]} tiles")

In [ ]:
def load_target(path, grid_w, grid_h, tile_px):
    if path is None:
        arr = data.astronaut()
        img = Image.fromarray(arr)
    else:
        img = Image.open(path)
        img = ImageOps.exif_transpose(img).convert('RGB')
    out_w, out_h = grid_w * tile_px, grid_h * tile_px
    src_w, src_h = img.size
    scale = min(out_w / src_w, out_h / src_h)
    new_w, new_h = int(src_w * scale), int(src_h * scale)
    resized = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
    canvas = Image.new('RGB', (out_w, out_h), (0, 0, 0))
    canvas.paste(resized, ((out_w - new_w) // 2, (out_h - new_h) // 2))
    return np.asarray(canvas, dtype=np.uint8)

target_rgb = load_target(
    CONFIG['TARGET_PATH'], CONFIG['GRID_W'], CONFIG['GRID_H'], CONFIG['TILE_PX']
)
target_lab = rgb2lab(target_rgb.astype(np.float32) / 255.0)
target_lab_grid = target_lab.reshape(
    CONFIG['GRID_H'], CONFIG['TILE_PX'],
    CONFIG['GRID_W'], CONFIG['TILE_PX'], 3
).mean(axis=(1, 3)).astype(np.float32)
target_lab_grid.shape

In [ ]:
idx = match.match_grid(
    target_lab_grid, pool.lab,
    lambda_=CONFIG['LAMBDA'], mu=CONFIG['MU'],
)
idx.shape

In [ ]:
img, usage = render.render_mosaic_with_usage(
    idx, pool, CONFIG['TILE_PX'], CONFIG['OUTPUT_PATH'],
    target_rgb=target_rgb, tone_strength=CONFIG['TAU'],
)
display(img)
print(f"\noutput written to {CONFIG['OUTPUT_PATH']}")
print(f"tiles used: {len(usage)} distinct / {sum(usage.values())} total placements")

## 交互滑条
拖动滑条不会自动重跑——点 **重跑** 按钮触发一次 match+render。

In [ ]:
lambda_slider = widgets.FloatSlider(value=CONFIG['LAMBDA'], min=0, max=50, step=0.5, description='λ (重复)', continuous_update=False)
mu_slider = widgets.FloatSlider(value=CONFIG['MU'], min=0, max=200, step=5, description='μ (邻居)', continuous_update=False)
tau_slider = widgets.FloatSlider(value=CONFIG['TAU'], min=0, max=1, step=0.05, description='τ (色调)', continuous_update=False)
rerun_btn = widgets.Button(description='重跑', button_style='primary')
out = widgets.Output()

def _rerun(_):
    with out:
        out.clear_output()
        idx = match.match_grid(
            target_lab_grid, pool.lab,
            lambda_=lambda_slider.value, mu=mu_slider.value,
        )
        img, usage = render.render_mosaic_with_usage(
            idx, pool, CONFIG['TILE_PX'], CONFIG['OUTPUT_PATH'],
            target_rgb=target_rgb, tone_strength=tau_slider.value,
        )
        display(img)
        print(f"tiles used: {len(usage)} distinct / {sum(usage.values())} total")

rerun_btn.on_click(_rerun)
display(widgets.VBox([lambda_slider, mu_slider, tau_slider, rerun_btn, out]))